# Week 53, Day 4: Grid DP

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w053d4.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
rows, cols = 3, 4
ways = [[1] * cols for _ in range(rows)]
for r in range(1, rows):
    for c in range(1, cols):
        ways[r][c] = ways[r - 1][c] + ways[r][c - 1]
for row in ways:
    print(row)

In [ ]:
def count_routes(grid):
    """grid: strings, '.' open, '#' blocked. Routes moving right and down."""
    rows, cols = len(grid), len(grid[0])
    ways = [[0] * cols for _ in range(rows)]
    for r in range(rows):
        for c in range(cols):
            if grid[r][c] == "#":
                continue
            if r == 0 and c == 0:
                ways[r][c] = 1
                continue
            above = ways[r - 1][c] if r > 0 else 0
            left = ways[r][c - 1] if c > 0 else 0
            ways[r][c] = above + left
    return ways[-1][-1]

print(count_routes(["...", ".#.", "..."]), count_routes([".#..", "....", "..#."]))

In [ ]:
def cheapest_route(cost):
    rows, cols = len(cost), len(cost[0])
    INF = float("inf")
    best = [[INF] * cols for _ in range(rows)]
    for r in range(rows):
        for c in range(cols):
            if r == 0 and c == 0:
                best[r][c] = cost[0][0]
                continue
            above = best[r - 1][c] if r > 0 else INF
            left = best[r][c - 1] if c > 0 else INF
            best[r][c] = cost[r][c] + min(above, left)
    moves, r, c = [], rows - 1, cols - 1
    while (r, c) != (0, 0):
        if c > 0 and (r == 0 or best[r][c - 1] <= best[r - 1][c]):
            moves.append("R")
            c -= 1
        else:
            moves.append("D")
            r -= 1
    return best[-1][-1], "".join(reversed(moves))

print(cheapest_route([[1, 3, 1], [1, 5, 1], [4, 2, 1]]))

In [ ]:
def count_routes_one_row(rows, cols):
    row = [1] * cols
    for _ in range(1, rows):
        for c in range(1, cols):
            row[c] += row[c - 1]       # old row[c] is "above"; row[c - 1] is "left"
    return row[-1]

print(count_routes_one_row(3, 4), count_routes_one_row(17, 17))

In [ ]:
def least_health(grid):
    rows, cols = len(grid), len(grid[0])
    INF = float("inf")
    need = [[INF] * (cols + 1) for _ in range(rows + 1)]
    need[rows][cols - 1] = need[rows - 1][cols] = 1        # just past the goal
    for r in range(rows - 1, -1, -1):
        for c in range(cols - 1, -1, -1):
            after = min(need[r + 1][c], need[r][c + 1])
            need[r][c] = max(1, after - grid[r][c])
    return need[0][0]

print(least_health([[-2, -3, 3], [-5, -10, 1], [10, 30, -5]]))

In [ ]:
def largest_square(grid):
    rows, cols = len(grid), len(grid[0])
    side = [[0] * cols for _ in range(rows)]
    for r in range(rows):
        for c in range(cols):
            if grid[r][c] == "1":
                if r == 0 or c == 0:
                    side[r][c] = 1
                else:
                    side[r][c] = 1 + min(side[r - 1][c], side[r][c - 1], side[r - 1][c - 1])
    return max(max(row) for row in side), sum(sum(row) for row in side)

print(largest_square(["10100", "10111", "11111", "10010"]))

> **Key idea.** Grid DP uses a cell as the state. With moves right and down, fill in row-major order: routes add the counts from above and the left, cheapest routes add the cell's cost to the smaller of the two, and walking back from the corner rebuilds the route. Treat cells outside the grid as 0 (for counts) or infinity (for costs) instead of special-casing edges, so obstacles on the edges work. One row of memory is enough when you don't need the route. When the answer depends on what comes later, as with the least starting health, fill backwards from the goal. The largest square uses 1 + the minimum of three neighbours.

### Exercise: Routes past the rocks

Write `count_routes(grid)`: the number of routes from the top-left to the bottom-right moving right and down, avoiding `#` cells (0 if the start or end is blocked). Then `count_routes_with(grid, through)`: routes that pass through the cell `through`, a `(row, col)` pair. (Routes to it, times routes from it to the end, which is the same count on the grid flipped both ways.)

In [ ]:
def count_routes(grid):
    ___

def count_routes_with(grid, through):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef brute(grid, through=None):\n    rows, cols = len(grid), len(grid[0])\n    count = [0]\n    def go(r, c, seen):\n        if r >= rows or c >= cols or grid[r][c] == \"#\":\n            return\n        seen = seen or (r, c) == through\n        if (r, c) == (rows - 1, cols - 1):\n            count[0] += seen\n            return\n        go(r + 1, c, seen)\n        go(r, c + 1, seen)\n    go(0, 0, through is None)\n    return count[0]\ndef test_routes():\n    \"routes\"\n    assert count_routes([\"...\", \".#.\", \"...\"]) == 2 and count_routes([\"#.\"]) == 0 and count_routes([\".\"]) == 1\n    assert count_routes([\".#..\", \"....\"]) == 1 and count_routes([\".\" * 10] * 10) == 48620\n    assert count_routes_with([\"...\", \"...\", \"...\"], (1, 1)) == 4\n    rng = random.Random(1)\n    for _ in range(100):\n        rows, cols = rng.randint(1, 5), rng.randint(1, 5)\n        grid = [\"\".join(\"#\" if rng.random() < 0.25 else \".\" for _ in range(cols)) for _ in range(rows)]\n        assert count_routes(grid) == brute(grid), grid\n        t = (rng.randrange(rows), rng.randrange(cols))\n        assert count_routes_with(grid, t) == brute(grid, t), (grid, t)")

### Exercise: The cheapest walk

Write `cheapest_route(cost)`: `(total, moves)`, the cheapest total of the cells visited from the top-left to the bottom-right (moving right `"R"` and down `"D"`), and the moves as a string. Any route of least cost is accepted. Then `cheapest_one_row(cost)`: the total only, keeping one row of memory.

In [ ]:
def cheapest_route(cost):
    ___

def cheapest_one_row(cost):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef brute(cost):\n    rows, cols = len(cost), len(cost[0])\n    best = [float(\"inf\")]\n    def go(r, c, total):\n        total += cost[r][c]\n        if (r, c) == (rows - 1, cols - 1):\n            best[0] = min(best[0], total)\n            return\n        if r + 1 < rows:\n            go(r + 1, c, total)\n        if c + 1 < cols:\n            go(r, c + 1, total)\n    go(0, 0, 0)\n    return best[0]\ndef walk(cost, moves):\n    r = c = 0\n    total = cost[0][0]\n    for m in moves:\n        r, c = (r + 1, c) if m == \"D\" else (r, c + 1)\n        total += cost[r][c]\n    return total, (r, c)\ndef test_cheapest():\n    \"costs\"\n    assert cheapest_route([[1, 3, 1], [1, 5, 1], [4, 2, 1]]) == (7, \"RRDD\")\n    assert cheapest_route([[5]]) == (5, \"\") and cheapest_one_row([[5]]) == 5\n    rng = random.Random(2)\n    for _ in range(100):\n        rows, cols = rng.randint(1, 6), rng.randint(1, 6)\n        cost = [[rng.randint(0, 9) for _ in range(cols)] for _ in range(rows)]\n        total, moves = cheapest_route(cost)\n        assert total == brute(cost) == cheapest_one_row(cost)\n        assert walk(cost, moves) == (total, (rows - 1, cols - 1))\n    big = [[(r * 7 + c * 3) % 10 for c in range(400)] for r in range(400)]\n    assert cheapest_one_row(big) == cheapest_route(big)[0]")

### Exercise: The least starting health

Write `least_health(grid)` by filling backwards from the goal: the least whole-number health to start with (at least 1) so that a walker going right and down from the top-left to the bottom-right, adding each cell's value (the start cell included), always has health of at least 1. Then `forwards_guess(grid)`: what you'd get by taking the route with the **largest total** (ties: any) and computing the health it needs; the checks show it can be worse.

In [ ]:
def least_health(grid):
    ___

def forwards_guess(grid):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef brute(grid):\n    rows, cols = len(grid), len(grid[0])\n    best = [float(\"inf\")]\n    def go(r, c, total, lowest):\n        total += grid[r][c]\n        lowest = min(lowest, total)\n        if (r, c) == (rows - 1, cols - 1):\n            best[0] = min(best[0], max(1, 1 - lowest))\n            return\n        if r + 1 < rows:\n            go(r + 1, c, total, lowest)\n        if c + 1 < cols:\n            go(r, c + 1, total, lowest)\n    go(0, 0, 0, 0)\n    return best[0]\ndef test_health():\n    \"backwards\"\n    assert least_health([[-2, -3, 3], [-5, -10, 1], [10, 30, -5]]) == 7\n    assert least_health([[0]]) == 1 and least_health([[-5]]) == 6 and least_health([[100]]) == 1\n    rng = random.Random(3)\n    worse = False\n    for _ in range(150):\n        rows, cols = rng.randint(1, 5), rng.randint(1, 5)\n        grid = [[rng.randint(-9, 9) for _ in range(cols)] for _ in range(rows)]\n        assert least_health(grid) == brute(grid), grid\n        guess = forwards_guess(grid)\n        assert guess >= least_health(grid)\n        worse = worse or guess > least_health(grid)\n    assert worse, \"the forwards guess is sometimes worse\"")

### Exercise: Squares of land

A map is a list of strings of `"1"` (land) and `"0"` (water). Write `largest_square(grid)`: the side of the largest all-land square, and `count_squares(grid)`: the number of all-land squares of every size.

In [ ]:
def largest_square(grid):
    ___

def count_squares(grid):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef brute(grid):\n    rows, cols = len(grid), len(grid[0])\n    sides = []\n    for r in range(rows):\n        for c in range(cols):\n            k = 1\n            while r + k <= rows and c + k <= cols and all(grid[i][j] == \"1\" for i in range(r, r + k) for j in range(c, c + k)):\n                sides.append(k)\n                k += 1\n    return max(sides, default=0), len(sides)\ndef test_squares():\n    \"squares\"\n    g = [\"10100\", \"10111\", \"11111\", \"10010\"]\n    assert largest_square(g) == 2 and count_squares(g) == 15\n    assert largest_square([\"0\"]) == 0 and count_squares([\"111\", \"111\", \"111\"]) == 14\n    rng = random.Random(4)\n    for _ in range(100):\n        rows, cols = rng.randint(1, 6), rng.randint(1, 6)\n        grid = [\"\".join(\"1\" if rng.random() < 0.7 else \"0\" for _ in range(cols)) for _ in range(rows)]\n        assert (largest_square(grid), count_squares(grid)) == brute(grid), grid")

### Exercise: Kings' routes

A king moves right, down, or diagonally down-right. Write `king_routes(rows, cols)`: the number of routes from the top-left to the bottom-right of an empty grid, using **one list** of length `cols` and a saved variable for the diagonal value. (These are the Delannoy numbers, named after Henri Delannoy, who studied them in the 1890s.)

In [ ]:
def king_routes(rows, cols):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("from functools import cache\ndef test_kings():\n    \"one row and a diagonal\"\n    assert king_routes(1, 1) == 1 and king_routes(2, 2) == 3 and king_routes(3, 3) == 13 and king_routes(4, 4) == 63\n    @cache\n    def d(r, c):\n        if r == 0 or c == 0:\n            return 1\n        return d(r - 1, c) + d(r, c - 1) + d(r - 1, c - 1)\n    for r in range(1, 12):\n        for c in range(1, 12):\n            assert king_routes(r, c) == d(r - 1, c - 1)\n    assert king_routes(1, 500) == 1 and king_routes(300, 300) > 10 ** 200")

### Bug hunt: Ones along the edge

This fills the first row and column with 1s before applying the recurrence, but the rock on the top edge means the cells to its right can't be reached along the top at all. It prints `10` instead of `4`. Treat every cell the same way, with cells outside the grid counting 0 and rocks counting 0.

In [ ]:
grid = [".#..", "....", "...."]
rows, cols = len(grid), len(grid[0])
ways = [[1] * cols if r == 0 else [1] + [0] * (cols - 1) for r in range(rows)]
for r in range(1, rows):
    for c in range(1, cols):
        ways[r][c] = 0 if grid[r][c] == "#" else ways[r - 1][c] + ways[r][c - 1]
print(ways[-1][-1])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"edges\"\n    expect_output(\"4\")")

### Bug hunt: The wrong neighbour

In the one-row version, the left neighbour must already hold **this** row's value. This loop runs right to left, so `row[c - 1]` still holds the row above, and it prints `4` instead of `10` routes for a 3 × 4 grid. Run left to right.

In [ ]:
rows, cols = 3, 4
row = [1] * cols
for _ in range(1, rows):
    for c in range(cols - 1, 0, -1):
        row[c] += row[c - 1]
print(row[-1])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"left to right\"\n    expect_output(\"10\")")

### Bug hunt: Health that goes to zero

This backwards fill forgets that health must stay **at least 1**: where a cell gives lots of health, the need drops to 0 or below, and it prints `-27` instead of `7`. Clamp each need with `max(1, ...)`.

In [ ]:
grid = [[-2, -3, 3], [-5, -10, 1], [10, 30, -5]]
rows, cols = 3, 3
INF = float("inf")
need = [[INF] * (cols + 1) for _ in range(rows + 1)]
need[rows][cols - 1] = need[rows - 1][cols] = 1
for r in range(rows - 1, -1, -1):
    for c in range(cols - 1, -1, -1):
        need[r][c] = min(need[r + 1][c], need[r][c + 1]) - grid[r][c]
print(need[0][0])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"at least one\"\n    expect_output(\"7\")")

### Puzzle: Largest square

Arrange the lines of the inner loop of the largest-square DP. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # routes across an empty 2 × 3 grid moving right and down
q = ___   # the fill order for least starting health: "forwards" or "backwards"
r = ___   # the value outside the grid for a cheapest-route DP (use float("inf") or 0)
s = ___   # the number of all-1 squares in ["11", "11"]
t = ___   # True or False: allowing moves up and left keeps the row-major fill order valid

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == 3 and q == \"backwards\" and r == float(\"inf\"), \"p, q and r\"\n    assert s == 5 and t is False, \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: two collectors

A grid holds coins: each cell has a whole number of coins (0 or more), or is a rock `-1`. Two collectors both start at the top-left and walk to the bottom-right, each moving right or down, at the same time. Each cell's coins can be collected only once, even if both walk through it. Write `most_coins(grid)`: the most coins they can collect together, or `0` if the end can't be reached.

Running the one-walker DP twice (best route, remove its coins, best route again) is **not** optimal; the checks include a grid where it fails. Instead, move both walkers one step at a time: after k steps each is on the diagonal r + c = k, so the state is `(k, r1, r2)`, and each step has four combinations of moves. Fill k upwards, keeping only the previous diagonal's table. Then `twice_greedy(grid)`: the two-pass approach, for comparison.

In [ ]:
def most_coins(grid):
    ___

def twice_greedy(grid):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import itertools, random\ndef routes(rows, cols):\n    for moves in set(itertools.permutations(\"R\" * (cols - 1) + \"D\" * (rows - 1))):\n        r = c = 0\n        cells = [(0, 0)]\n        for m in moves:\n            r, c = (r + 1, c) if m == \"D\" else (r, c + 1)\n            cells.append((r, c))\n        yield cells\ndef brute(grid):\n    rows, cols = len(grid), len(grid[0])\n    ok = [p for p in routes(rows, cols) if all(grid[r][c] != -1 for r, c in p)]\n    best = 0\n    for a in ok:\n        for b in ok:\n            best = max(best, sum(grid[r][c] for r, c in set(a) | set(b)))\n    return best\ndef test_two():\n    \"two walkers\"\n    assert most_coins([[0, 1, -1], [1, 0, -1], [1, 1, 1]]) == 5\n    assert most_coins([[1, 1, -1], [1, -1, 1], [-1, 1, 1]]) == 0\n    assert most_coins([[7]]) == 7\n    trap = [[1, 1, 1, 1, 0, 0, 0], [0, 0, 0, 1, 0, 0, 0], [0, 0, 0, 1, 0, 0, 1], [1, 0, 0, 1, 0, 0, 0], [0, 0, 0, 1, 0, 0, 0], [0, 0, 0, 1, 0, 0, 0], [0, 0, 0, 1, 1, 1, 1]]\n    assert most_coins(trap) == 15 and twice_greedy(trap) < 15\n    rng = random.Random(5)\n    for _ in range(40):\n        rows, cols = rng.randint(1, 4), rng.randint(1, 4)\n        grid = [[-1 if rng.random() < 0.15 else rng.randint(0, 5) for _ in range(cols)] for _ in range(rows)]\n        grid[0][0], grid[-1][-1] = rng.randint(0, 5), rng.randint(0, 5)\n        assert most_coins(grid) == brute(grid), grid\n        assert twice_greedy(grid) <= most_coins(grid)\n    big = [[(r * 5 + c * 3) % 7 for c in range(40)] for r in range(40)]\n    assert most_coins(big) >= twice_greedy(big)")

## Warm-up

Each cell's count is the sum of the cell above and the cell to the left, because the last move into it came from one of them. The first row and column have only one route each: straight along the edge. The numbers are Pascal's triangle on its side: there are C(5, 2) = 10 routes to the far corner of a 3 × 4 grid.

## Counting routes

Moving only **right** and **down** from the top-left, define

**ways(r, c) = the number of routes from the start to cell (r, c)**,

with ways(0, 0) = 1. The last move came from above or from the left, so ways(r, c) = ways(r − 1, c) + ways(r, c − 1), with cells outside the grid counting 0. Both ingredients come earlier in **row-major order** (row by row, left to right), so that's the fill order.

**Obstacles** fit naturally: a blocked cell has 0 routes. Don't fill the first row and column with 1s blindly: a rock on the top edge cuts off every cell beyond it. Handling the edges by "0 if outside the grid" rather than by special cases avoids that bug.

## The cheapest route

With a cost in each cell, the same shape gives the cheapest route:

**best(r, c) = cost(r, c) + min(best(r − 1, c), best(r, c − 1))**,

with infinity outside the grid (so the min ignores it) and best(0, 0) = cost(0, 0). To recover the route, walk back from the corner: at each cell, step to whichever neighbour (above or left) has the smaller best, and reverse the list of moves at the end.

This is a shortest-path problem, and Dijkstra (Week 51) would solve it too. DP is simpler and faster here because moves only go right and down, so there's a fill order with no cycles. Allow moves up or left and that order vanishes: then it's Dijkstra's job.

## One row is enough

Each row only reads the row above and itself. So keep **one list** of length `cols`: before you overwrite `row[c]`, it still holds the value from the row above (`best(r − 1, c)`), and `row[c − 1]` already holds the new value from this row (`best(r, c − 1)`). Memory drops from rows × cols to cols, which matters for wide, tall grids, at the usual price: no route reconstruction.

If the moves include the **diagonal** (down and right), the cell diagonally up-left has already been overwritten by the time you reach `row[c]`, so save it in a variable before overwriting `row[c − 1]`.

## Filling from the goal

Sometimes the answer at a cell depends on what comes **after** it, not before. A knight walks from the top-left to the bottom-right, moving right and down; each cell adds to or takes from their health, which must stay at least 1 the whole way. What's the least health to start with?

Filling forwards fails: the best route so far might be the one that later walks into a disaster. Instead define

**need(r, c) = the least health needed on entering (r, c) to finish alive**,

which depends on the cells **after** it: need(r, c) = max(1, min(need(r + 1, c), need(r, c + 1)) − cell(r, c)), and at the goal max(1, 1 − cell). Fill from the bottom-right corner, backwards. (Puzzles like this one are widely shared as "the dungeon problem".)

## Squares and shapes

Grid DP isn't only about routes. To find the **largest square of 1s** in a grid of 0s and 1s, define

**side(r, c) = the side of the largest all-1 square whose bottom-right corner is (r, c)**.

If the cell is 0, it's 0. Otherwise it's 1 + min(side(r − 1, c), side(r, c − 1), side(r − 1, c − 1)): the square can grow only as far as all three neighbouring squares allow. It's "best ending here" from yesterday, in two dimensions, and the answer is the maximum over all cells. A bonus: the sum of side(r, c) over every cell counts **all** the all-1 squares, because a square of side k ending at a cell means one square of each side 1 to k ends there.

## Practice

## Quiz

## Daily challenge

## Before you go

Sketch a 4 × 4 grid with two rocks and fill in the route counts by hand, then check with your `count_routes`. Tomorrow: the knapsack family, the most useful DP of all.